# Online Retail II Customer Sales Analysis

## DQA-01 | Objectives

Author: Yoonju Song

Date: July 2026

The purpose of this notebook is to evaluate the quality of the Online Retail II dataset before performing exploratory data analysis and business analysis.

The assessment includes:

- Dataset overview
- Data types
- Missing values
- Duplicate records
- Invalid values
- Initial observations


## DQA-02 | Environment & Data Loading

This section initializes the Python analysis environment and loads the Online Retail II dataset.

**Objectives:**
- Import the required Python libraries.
- Load the raw transaction dataset.
- Verify the dataset structure and dimensions.

In [3]:
# DQA-02-01: Import Libraries and Configure Environment

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
# DQA-02-02: Load Raw Dataset

df = pd.read_csv("../data/raw/online_retail_II.csv")

In [5]:
# DQA-02-03: Initial Dataset Inspection

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:")
display(df.head())

print("\nDataset Information:")
df.info()

Dataset Shape: (1067371, 8)

First 5 Rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


## DQA-03 | Dataset Overview

This section examines the dataset dimensions, column structure, and data types before assessing data quality.
The dataset contains **1,067,371 transaction records** and **8 columns**.

Initial observations:

- Invoice, StockCode, Quantity, InvoiceDate and Price contain no missing values.
- Description contains missing values.
- Data cleaning will be required before performing business analysis

In [6]:
# DQA-03-01: Dataset Structure Summary

dataset_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Non-Null Count": df.notna().sum(),
    "Missing Count": df.isna().sum()
})

display(dataset_summary)

print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")

,Data Type,Non-Null Count,Missing Count
Invoice,str,1067371,0
StockCode,str,1067371,0
Description,str,1062989,4382
Quantity,int64,1067371,0
InvoiceDate,str,1067371,0
Price,float64,1067371,0
Customer ID,float64,824364,243007
Country,str,1067371,0


Total Rows: 1,067,371
Total Columns: 8


### Interpretation

- The dataset contains transaction-level records with information on invoices, products, quantities, prices, customers, and countries.
- Each row represents a transaction line rather than a unique customer or order.
- Data types and missing values will be reviewed before applying cleaning rules.

## DQA-04 | Missing Value Assessment

This section evaluates the completeness of each variable and identifies missing-value risks for downstream analysis.

In [7]:
# DQA-04-01: Missing Value Summary

missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage (%)": (
        df.isna().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)

display(missing_summary)

,Missing Count,Missing Percentage (%)
Customer ID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


### Interpretation

- **Customer ID:** 243,007 missing values (22.77%). These records require special handling because customer-level analysis and RFM segmentation depend on identifiable customers.
- **Description:** 4,382 missing values (approximately 0.41%). Missing product descriptions may limit product-level interpretation.
- **Other variables:** No missing values were detected in the remaining six columns.

**Cleaning Implication:** Apply missing-value rules according to the analytical objective. Customer ID is required for RFM analysis, but its absence does not automatically make a transaction invalid for every sales analysis.

## DQA-05 | Duplicate Records Assessment

This section identifies exact duplicate records and evaluates their proportion in the raw dataset before data cleaning.

In [8]:
# DQA-05-01: Duplicate Records Assessment

duplicate_count = df.duplicated().sum()
duplicate_percentage = round(
    duplicate_count / len(df) * 100, 2
)

print(f"Total Records: {len(df):,}")
print(f"Duplicate Records: {duplicate_count:,}")
print(f"Duplicate Percentage: {duplicate_percentage:.2f}%")

Total Records: 1,067,371
Duplicate Records: 34,335
Duplicate Percentage: 3.22%


### Interpretation

- **34,335 records (3.22%)** were identified as exact duplicates across all columns.
- Duplicate records may lead to overestimation of transaction-level metrics and revenue.
- Exact duplicates will be removed during Data Cleaning, following the project's established cleaning rules.

**Cleaning Implication:** Verify that duplicate removal is applied consistently before calculating sales performance and customer-level metrics.

## DQA-06 | Data Type Validation

This section reviews the data types of each column and identifies variables that require conversion during data cleaning.

In [9]:
# DQA-06-01: Data Type Inspection

dtype_summary = pd.DataFrame({
    "Column": df.columns,
    "Current Data Type": df.dtypes.astype(str).values,
    "Missing Count": df.isna().sum().values
})

display(dtype_summary)

print("\nInvoiceDate Sample:")
display(df["InvoiceDate"].head())

,Column,Current Data Type,Missing Count
0,Invoice,str,0
1,StockCode,str,0
2,Description,str,4382
3,Quantity,int64,0
4,InvoiceDate,str,0
5,Price,float64,0
6,Customer ID,float64,243007
7,Country,str,0



InvoiceDate Sample:


0    2009-12-01 07:45:00
1    2009-12-01 07:45:00
2    2009-12-01 07:45:00
3    2009-12-01 07:45:00
4    2009-12-01 07:45:00
Name: InvoiceDate, dtype: str

### Interpretation

- **InvoiceDate:** Currently stored as a string and requires conversion to datetime for time-series analysis and RFM Recency calculation.
- **Customer ID:** Stored as float64 due to missing values. Customer identifiers should be handled consistently during data cleaning.
- **Quantity and Price:** Stored as numeric types and will be checked for invalid values in subsequent validation steps.
- **Other variables:** Text-based identifiers and categorical fields should be preserved appropriately.

**Cleaning Implication:** Convert InvoiceDate to datetime and establish consistent Customer ID handling rules before downstream analysis.


## DQA-07 | Quantity Validation

This section examines negative and zero quantities and evaluates their relationship with cancellation indicators in invoice numbers.

In [10]:
# DQA-07-01: Quantity and Cancellation Validation

# Identify negative and zero quantities
negative_quantity = df[df["Quantity"] < 0].copy()
zero_quantity = df[df["Quantity"] == 0].copy()

# Identify cancellation indicators in invoice numbers
cancel_flag = df["Invoice"].astype("string").str.upper().str.startswith("C").fillna(False)

# Compare negative quantities with cancellation indicators
negative_cancel = cancel_flag.loc[negative_quantity.index]

print(f"Total Records: {len(df):,}")
print(f"Negative Quantity Records: {len(negative_quantity):,}")
print(f"Zero Quantity Records: {len(zero_quantity):,}")
print(f"Cancellation-Flagged Records: {cancel_flag.sum():,}")

print(f"Negative Quantity + Cancellation Flag: {negative_cancel.sum():,}")
print(f"Negative Quantity without Cancellation Flag: {(~negative_cancel).sum():,}")

print("\nSample Negative Quantity Records:")
display(negative_quantity.head())

Total Records: 1,067,371
Negative Quantity Records: 22,950
Zero Quantity Records: 0
Cancellation-Flagged Records: 19,494
Negative Quantity + Cancellation Flag: 19,493
Negative Quantity without Cancellation Flag: 3,457

Sample Negative Quantity Records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


### Interpretation

- **Negative Quantity:** 22,950 records (2.15%) contain negative quantities.
- **Zero Quantity:** No records contain zero quantities.
- **Cancellation Indicator:** 19,494 records have invoice numbers beginning with "C".
- **Cancellation Relationship:** Among negative-quantity records, 19,493 (84.94%) have cancellation indicators, while 3,457 (15.06%) do not.
- **Data Quality Concern:** Negative quantities without cancellation indicators require further investigation. Their business meaning cannot be determined from the available fields alone.

**Cleaning Implication:** For positive-sales analysis, apply explicit quantity and cancellation filters. Review records with negative quantities separately rather than assuming that all represent cancellations.

## DQA-08 | Price Validation

This section examines zero and negative prices to identify records that may affect revenue calculations and downstream sales analysis.

In [11]:
# DQA-08-01: Price Validation

# Classify price values
negative_price = df[df["Price"] < 0].copy()
zero_price = df[df["Price"] == 0].copy()
positive_price = df[df["Price"] > 0].copy()

# Summarize price validation
price_summary = pd.DataFrame({
    "Price Category": [
        "Negative Price",
        "Zero Price",
        "Positive Price",
        "Total"
    ],
    "Record Count": [
        len(negative_price),
        len(zero_price),
        len(positive_price),
        len(df)
    ]
})

price_summary["Percentage (%)"] = (
    price_summary["Record Count"] / len(df) * 100
).round(2)

display(price_summary)

print(f"Price <= 0 Records: {len(negative_price) + len(zero_price):,}")

print("\nSample Negative Price Records:")
display(negative_price.head())

print("\nSample Zero Price Records:")
display(zero_price.head())

,Price Category,Record Count,Percentage (%)
0,Negative Price,5,0.00
1,Zero Price,6202,0.58
2,Positive Price,1061164,99.42
3,Total,1067371,100.00


Price <= 0 Records: 6,207

Sample Negative Price Records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom



Sample Zero Price Records:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom


### Interpretation

- **Negative Price:** 5 records (approximately 0.0005%) have negative prices. All five are labeled "Adjust bad debt" and appear to represent accounting adjustments rather than ordinary product sales.
- **Zero Price:** 6,202 records (approximately 0.58%) have a price of zero. Sample records include negative quantities and adjustment-like descriptions, so these records should not automatically be interpreted as free products or promotions.
- **Positive Price:** 1,061,164 records (approximately 99.42%) have positive prices.
- **Data Quality Concern:** Non-positive prices may affect revenue calculations and require explicit treatment based on the analytical objective.

**Cleaning Implication:** For positive-sales analysis, apply `Price > 0` together with appropriate quantity and cancellation filters. Preserve the original records for auditability and document excluded transactions separately.